# Lab 17: Retrieval evaluation and the isolation test

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/sailokeshreddynandavarapu/Documents/Sem_5_Fall_26/Lab_Activity/Lab2/DAMG-7245-Project-Lantern-Case-Study-Lab2-Team-2 | LLM mode: replay


In [2]:
import numpy as np, shutil
from src import sample_lsa
recs = sd.records("deal_a") + sd.records("deal_b")
chunks = [{"chunk_id": f'{r["doc_id"]}:{r["block_id"]}', "deal_id": r["deal_id"], "doc_id": r["doc_id"], "tier": r["tier"],
           "block_ids": [r["block_id"]], "text": r["text"]} for r in recs]
vecs = llm.embed([c["text"] for c in chunks], sample_lsa.MODEL, compute=sample_lsa.encode).tolist()
embed = lambda text: llm.embed([text], sample_lsa.MODEL, compute=sample_lsa.encode)[0].tolist()
import chromadb                    # rerun-safe: drop the collection through the client, not the folder
try:
    chromadb.PersistentClient(path="data/index").delete_collection("lantern")
except Exception:
    pass                           # first run: nothing to delete

In [3]:
# From the slide: A vector store with scope enforced in the store
import chromadb, json

client = chromadb.PersistentClient(path="data/index")
col = client.get_or_create_collection(
    "lantern", configuration={"hnsw": {"space": "cosine"}})

col.add(ids=[c["chunk_id"] for c in chunks],
        embeddings=vecs,
        documents=[c["text"] for c in chunks],
        metadatas=[{"deal_id": c["deal_id"],
                    "doc_id": c["doc_id"],
                    "tier": c["tier"],
                    "block_ids": json.dumps(c["block_ids"])}
                   for c in chunks])

def search(qvec, scope: str, k: int = 10):
    if not scope:
        raise ValueError("scope is required")
    return col.query(query_embeddings=[qvec], n_results=k,
        where={"deal_id": {"$in": [scope, "public"]}})

In [4]:
# From the slide: BM25 plus dense, fused with reciprocal rank fusion
from rank_bm25 import BM25Okapi
tok = lambda s: s.lower().split()
bm25 = BM25Okapi([tok(c["text"]) for c in chunks])

def rrf(*rankings, k=60):
    score = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            score[doc] = (score.get(doc, 0)
                          + 1 / (k + rank))
    return sorted(score, key=score.get, reverse=True)

def hybrid(query, qvec, scope, k=10):
    dense = search(qvec, scope, k=50)["ids"][0]
    s = bm25.get_scores(tok(query))
    ok = (scope, "public")         # scope BM25 too
    lex = [chunks[i]["chunk_id"]
           for i in s.argsort()[::-1]
           if chunks[i]["deal_id"] in ok][:50]
    return rrf(dense, lex)[:k]

In [5]:
# From the slide: The metrics, implemented and tested
import math

def recall_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & set(rel)) / len(rel)

def mrr(ranked, rel):
    for i, d in enumerate(ranked, 1):
        if d in rel:
            return 1 / i
    return 0.0

def ndcg_at_k(ranked, grades, k):
    gain = lambda g, i: (2**g - 1) / math.log2(i + 1)
    dcg = sum(gain(grades.get(d, 0), i)
              for i, d in enumerate(ranked[:k], 1))
    ideal = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(gain(g, i)
               for i, g in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0

In [6]:
import pandas as pd
qrels = [json.loads(l) for l in open("data/eval/qrels.jsonl")]
rows = []
for q in qrels:
    rel, grades = list(q["relevant"]), q["relevant"]
    dense = search(embed(q["query"]), q["scope"], k=20)["ids"][0]
    hyb = hybrid(q["query"], embed(q["query"]), q["scope"], k=20)
    for name, ranked in (("dense", dense), ("hybrid", hyb)):
        rows.append({"retriever": name, "category": q["category"], "recall@5": recall_at_k(ranked, rel, 5),
                     "mrr": mrr(ranked, rel), "ndcg@10": ndcg_at_k(ranked, grades, 10)})
pd.DataFrame(rows).groupby("retriever")[["recall@5", "mrr", "ndcg@10"]].mean().round(3)

,recall@5,mrr,ndcg@10
retriever,,,
dense,1.000,1.000,0.958
hybrid,0.917,0.903,0.890


The isolation test from the slide runs unchanged: in this repository `data/eval/qrels.jsonl` and `data/dataroom/canaries_visible.json` are where Case Study 2 puts them.

In [7]:
chunk_meta = {c["chunk_id"]: c for c in chunks}.get

In [8]:
# From the slide: The isolation test, and proving it has teeth
import json, os, pytest
ROOT = os.environ.get("LANTERN_ISOLATION_FIXTURE", "data")
QRELS = f"{ROOT}/eval/qrels.jsonl"
CANS = f"{ROOT}/dataroom/canaries_visible.json"
QUERIES = [json.loads(l) for l in open(QRELS)]
CANARIES_B = json.load(open(CANS))["deal_b"]

@pytest.mark.parametrize("q", QUERIES)
def test_deal_a_never_sees_deal_b(q):
    hits = hybrid(q["query"], embed(q["query"]),
                  scope="deal_a", k=20)
    for cid in hits:
        meta = chunk_meta(cid)
        assert meta["deal_id"] in ("deal_a", "public")
        text = meta["text"]
        assert not any(c in text for c in CANARIES_B)

In [9]:
for q in QUERIES: test_deal_a_never_sees_deal_b(q)
print("isolation test passes on", len(QUERIES), "queries")
# Teeth: remove the scope filter and watch it fail
_safe = hybrid
def hybrid(query, qvec, scope, k=10):
    return [c["chunk_id"] for c in chunks][:k] + [c["chunk_id"] for c in chunks if c["deal_id"] == "deal_b"]
try:
    for q in QUERIES: test_deal_a_never_sees_deal_b(q)
    print("NO FAILURE: the test has no teeth")
except AssertionError:
    print("with the filter removed, the test fails: it has teeth")
hybrid = _safe

isolation test passes on 12 queries
with the filter removed, the test fails: it has teeth


**Exercise 17 — label three management-call queries before scoring.**
The three labels below were chosen from the transcript and CIM: one numeric lookup, one cross-document comparison, and one data-room claim. Grade 2 denotes direct answer evidence; grade 1 denotes supporting context. `w-13` needs both the CFO explanation and the CIM claim. `w-14` asks specifically for the CFO’s trailing figure, so the conflicting CIM percentage is not relevant. `w-15` uses the CEO’s statement directly and the CIM’s concentration claim as supporting context.
The previous exercise contained three cross-document queries. The numeric and claim queries replace `w-14` and `w-15`; these revised labels are saved before the evaluation cell runs. The original 12 sample qrels remain unchanged.

In [10]:
new_qrels = [
    {
        "qid": "w-13",
        "query": "How does the CFO's trailing net revenue retention explanation compare with the CIM's 118% claim?",
        "category": "cross_document",
        "scope": "deal_a",
        "relevant": {
            "DR-deal_a-002:t003": 2,
            "DR-deal_a-001:p0003_b002": 2
        }
    },
    {
        "qid": "w-14",
        "query": "According to the CFO in the management call, what is net revenue retention for the trailing twelve months?",
        "category": "numeric_lookup",
        "scope": "deal_a",
        "relevant": {
            "DR-deal_a-002:t003": 2
        }
    },
    {
        "qid": "w-15",
        "query": "What does the CEO claim about dependence on the largest customer in the management call?",
        "category": "data_room_claim",
        "scope": "deal_a",
        "relevant": {
            "DR-deal_a-002:t002": 2,
            "DR-deal_a-001:p0003_b001": 1
        }
    }
]

assert {q["category"] for q in new_qrels} == {"numeric_lookup", "cross_document", "data_room_claim"}
assert len({q["qid"] for q in new_qrels}) == 3
chunk_by_id = {c["chunk_id"]: c for c in chunks}
for q in new_qrels:
    assert q["relevant"]
    for cid, grade in q["relevant"].items():
        assert cid in chunk_by_id and grade in (1, 2)
        assert chunk_by_id[cid]["deal_id"] == q["scope"]
exercise_path = Path("data/eval/qrels_exercise17.jsonl")
exercise_path.write_text("\n".join(json.dumps(q) for q in new_qrels) + "\n")
print("Saved one numeric, one cross-document, and one data-room claim query before scoring.")


Saved one numeric, one cross-document, and one data-room claim query before scoring.


In [11]:
# Cache embeddings for the new questions locally.
previous_mode = llm.MODE
try:
    llm.MODE = "record"
    llm.embed(
        [q["query"] for q in new_qrels],
        sample_lsa.MODEL,
        compute=sample_lsa.encode,
    )
finally:
    llm.MODE = previous_mode

original_qrels = [
    json.loads(line)
    for line in Path("data/eval/qrels.jsonl").read_text().splitlines()
    if line.strip()
]

# Merge by ID so rerunning does not duplicate questions.
all_qrels = {
    q["qid"]: q for q in original_qrels + new_qrels
}.values()

results = []
for q in all_qrels:
    qvec = embed(q["query"])
    lexical_scores = bm25.get_scores(tok(q["query"]))
    lexical = [chunks[i]["chunk_id"] for i in lexical_scores.argsort()[::-1]
               if chunks[i]["deal_id"] in (q["scope"], "public")]
    rankings = {
        "dense": search(qvec, q["scope"], k=20)["ids"][0],
        "hybrid": hybrid(q["query"], qvec, q["scope"], k=20),
        "bm25": lexical,
    }
    for name, ranked in rankings.items():
        assert all(chunk_by_id[cid]["deal_id"] in (q["scope"], "public") for cid in ranked)
        results.append({
            "qid": q["qid"], "category": q["category"], "retriever": name,
            "recall@5": recall_at_k(ranked, q["relevant"], 5),
            "mrr": mrr(ranked, q["relevant"]),
            "ndcg@10": ndcg_at_k(ranked, q["relevant"], 10),
        })

assert len(all_qrels) == 15
details = pd.DataFrame(results)
metrics = ["recall@5", "mrr", "ndcg@10"]
summary = details.groupby(["category", "retriever"])[metrics].mean()
print("Mean results by category over all 15 queries:")
display(summary.round(3))
print("Results for the three exercise queries:")
display(details[details["qid"].isin([q["qid"] for q in new_qrels])].round(3))
cross = summary.loc["cross_document"]
print("Cross-document summary:")
display(cross.round(3))
best = cross["ndcg@10"].max()
winners = cross.index[np.isclose(cross["ndcg@10"], best)].tolist()
print("Cross-document winner(s) by mean nDCG@10:", ", ".join(winners))


Mean results by category over all 15 queries:


recall@5    mrr  ndcg@10
category        retriever                          
cross_document  bm25          0.833  1.000    0.849
                dense         1.000  1.000    0.978
                hybrid        0.833  1.000    0.914
data_room_claim bm25          1.000  0.875    0.856
                dense         0.875  0.875    0.851
                hybrid        0.875  0.875    0.854
definitional    bm25          1.000  1.000    1.000
                dense         1.000  1.000    1.000
                hybrid        1.000  1.000    1.000
numeric_lookup  bm25          0.750  0.786    0.839
                dense         1.000  1.000    0.940
                hybrid        0.875  0.833    0.878
section_seeking bm25          1.000  1.000    1.000
                dense         1.000  1.000    1.000
                hybrid        1.000  1.000    1.000
table_seeking   bm25          1.000  0.333    0.500
                dense         1.000  1.000    1.000
                hybrid        1.000  0.500    0.631

Results for the three exercise queries:


,qid,category,retriever,recall@5,mrr,ndcg@10
36,w-13,cross_document,dense,1.0,1.0,1.000
37,w-13,cross_document,hybrid,1.0,1.0,1.000
38,w-13,cross_document,bm25,1.0,1.0,1.000
39,w-14,numeric_lookup,dense,1.0,1.0,1.000
40,w-14,numeric_lookup,hybrid,1.0,1.0,1.000
41,w-14,numeric_lookup,bm25,1.0,1.0,1.000
42,w-15,data_room_claim,dense,0.5,0.5,0.608
43,w-15,data_room_claim,hybrid,0.5,0.5,0.619
44,w-15,data_room_claim,bm25,1.0,0.5,0.628


Cross-document summary:


,recall@5,mrr,ndcg@10
retriever,,,
bm25,0.833,1.0,0.849
dense,1.000,1.0,0.978
hybrid,0.833,1.0,0.914


Cross-document winner(s) by mean nDCG@10: dense


**Interpretation and limits.** Use the cross-document summary to compare how highly each retriever places the graded evidence. MRR alone can tie even when a retriever misses the second document; recall and nDCG expose that difference. Dense retrieval has an advantage on this sample because its local LSA embedder was fitted on these texts. Inspect the three new-query rows separately from the category averages, which include original sample questions.
If recall@10 were 0.9 but answers were still wrong, inspect evidence selection, generation, source attribution, and numeric linting next. High retrieval recall does not guarantee a faithful answer. These 15 classroom queries are not the separate tutorial’s 20-query strategy comparison or the CS2 requirement of at least 60 qrels.

**Measured conclusion:** Dense wins across the three cross-document queries with mean nDCG@10 0.978, versus hybrid 0.914 and BM25 0.849. Dense also retrieves all relevant blocks within the top five (recall@5 1.000); hybrid and BM25 average 0.833. However, the new cross-document question `w-13` is a three-way tie at 1.000 on every metric: it does not independently establish a dense advantage. Its explicit NRR wording helps all three methods retrieve the two labelled passages. The new numeric question also ties; BM25 retrieves more relevant evidence for the new management-claim question. Performance therefore depends on the question category, not just the overall winner.
